In [2]:
# 03_guardrail_agent.ipynb
# =============================================================================
# Personalized Health Intervention Pathways for Comorbid Policyholders:
# Loss Ratio Management via LLM Guardrails and Counterfactual Explanations
# -----------------------------------------------------------------------------
# Notebook 03 — GPT-4o-mini Clinical Guardrail Agent (Multi-Case)
# Input  : ../results/tables/agent_config.pkl
#          ../results/tables/df_final.pkl
#          ../results/tables/model_{group}.pkl
# Output : ../results/tables/guardrail_ranges.json
#          (keys: "{group_name}_case{1|2|3}")
# =============================================================================

# %%
# ## 0. Import Libraries

import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
from openai import OpenAI
from dotenv import load_dotenv

warnings.filterwarnings('ignore')

# %%
# ## 1. Load Models and Configuration

agent_config  = joblib.load('../results/tables/agent_config.pkl')
df_final      = joblib.load('../results/tables/df_final.pkl')

X_FEATURES      = agent_config['X_features']
NUM_FEATURES    = agent_config['num_features']
CAT_FEATURES    = agent_config['cat_features']
VARY_FEATURES   = agent_config['vary_features']
FIXED_FEATURES  = agent_config['fixed_features']
TARGET_COL      = agent_config['target_col']
AGEGROUP_CONFIG = agent_config['agegroup_config']

models = {}
for group_name in AGEGROUP_CONFIG:
    try:
        models[group_name] = joblib.load(
            f'../results/tables/model_{group_name}.pkl'
        )
    except FileNotFoundError:
        print(f"WARNING: model_{group_name}.pkl not found.")
        models[group_name] = None

print("Models loaded:", [k for k, v in models.items() if v is not None])

# %%
# ## 2. OpenAI Client Setup

load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
LLM_MODEL      = os.getenv('LLM_MODEL', 'gpt-4o-mini')

if not OPENAI_API_KEY:
    raise ValueError("[ERROR] OPENAI_API_KEY not set in .env")

client = OpenAI(api_key=OPENAI_API_KEY)
print(f"LLM model: {LLM_MODEL}")

# %%
# ## 3. Representative Case Index Selection
# Three typical Class 3 patients per group selected by proximity to group median
# Indices identified in exploratory analysis (see notebook 05)

CASE_INDICES = {
    "MiddleAged_Male":   [3972, 5762, 4265],
    "MiddleAged_Female": [5903, 1561, 5323],
    "Older_Male":        [4793, 1673, 4989],
    "Older_Female":      [4918, 3010, 3991],
}

# %%
# ## 4. LLM-Based Clinical Guardrail Agent

def get_clinical_guardrails(patient_data: pd.DataFrame,
                             group_name: str,
                             allowed_features: list,
                             metadata: dict) -> dict:
    """
    Call GPT-4o-mini to derive patient-specific clinical reasoning.
    LLM output is stored as reasoning log only;
    final permitted ranges are determined by Hard Rules A-E.

    Parameters
    ----------
    patient_data     : single-row DataFrame with current indicator values
    group_name       : stratified group label (e.g. 'Older_Male')
    allowed_features : list of feature names
    metadata         : dict of valid values per categorical feature

    Returns
    -------
    dict with keys 'reasoning' (str) and 'guardrail_ranges' (dict)
    """
    patient_info = patient_data.to_dict(orient='records')[0]

    prompt = f"""
You are a high-precision Clinical Data Strategist for a health insurance company.
You must respond ONLY in valid JSON using the exact variable names listed below.

[Patient Information ({group_name})]
{json.dumps(patient_info, ensure_ascii=False, indent=1)}

[Available Variable Names]
{", ".join(allowed_features)}

[Categorical Variable Valid Values]
{json.dumps(metadata, ensure_ascii=False, indent=1)}

[Guardrail Principles — apply ALL strictly]

1. CONFLICT PREVENTION:
   - Cap Carb_g at [0, current Carb_g] and Sugar_g at [0, current Sugar_g]
     whenever Sodium_mg is reduced below its current value.

2. PHYSICAL CONSISTENCY:
   - BMI, WaistCirc, Weight: all must move in the same direction.
   - Each bounded to [current * 0.85, current].

3. ENERGY FLOOR:
   - Energy_kcal: [max(current * 0.70, 500), current].

4. SAFETY: All lower bounds >= 0.

5. DIVERSITY HINT:
   - Allow Fat_g to vary in both directions for structural pathway diversity.

[Response Format — valid JSON only, no extra text]
{{
    "reasoning": "Detailed clinical rationale for each constraint",
    "guardrail_ranges": {{ "ExactVariableName": [min, max] }}
}}
"""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert who responds strictly in valid JSON, "
                    "fully adhering to clinical guidelines and the provided "
                    "data schema."
                )
            },
            {"role": "user", "content": prompt},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

# %%
# ## 5. Hard Rules A-E (Applied Directly from Patient Baseline)

def apply_hard_rules(query: pd.DataFrame, features: list) -> dict:
    """
    Derive permitted ranges directly from patient baseline values.
    LLM output is NOT used here — rules applied unconditionally.

    Rule A — Anthropometric consistency : [0.85c, c]
    Rule B — Energy floor and ceiling   : [max(0.70c, 500), c]
    Rule C — Sodium bounds              : [max(0.60c, 800), 0.90c]
    Rule D — Carb / sugar ceiling       : ceiling = c
    Rule E — Macronutrient floors       :
        Protein_g    >= max(0.60c, 30)
        Potassium_mg >= max(0.50c, 500)
        Carb_g       >= max(0.20c, 30)
        Fiber_g      >= max(0.30c, 5)
    Default for all other features      : [max(0.70c, 0), 1.30c]
    """
    cur = {f: float(query.iloc[0][f]) for f in features}
    g   = {}

    # Default range
    for f in features:
        g[f] = [round(max(cur[f] * 0.70, 0.0), 4),
                round(cur[f] * 1.30, 4)]

    # Rule A
    for f in ['BMI', 'WaistCirc', 'Weight']:
        g[f] = [round(cur[f] * 0.85, 4), round(cur[f], 4)]
    wt_ratio = g['Weight'][0] / cur['Weight']
    g['WaistCirc'][0] = max(g['WaistCirc'][0],
                            round(cur['WaistCirc'] * wt_ratio, 4))
    g['BMI'][0]       = max(g['BMI'][0],
                            round(cur['BMI']       * wt_ratio, 4))

    # Rule B
    g['Energy_kcal'] = [round(max(cur['Energy_kcal'] * 0.70, 500.0), 4),
                        round(cur['Energy_kcal'], 4)]

    # Rule C
    g['Sodium_mg'] = [round(max(cur['Sodium_mg'] * 0.60, 800.0), 4),
                      round(cur['Sodium_mg'] * 0.90, 4)]

    # Rule D
    g['Carb_g'][1]  = round(cur['Carb_g'],  4)
    g['Sugar_g'][1] = round(cur['Sugar_g'], 4)

    # Rule E
    g['Protein_g'][0]    = round(max(cur['Protein_g']    * 0.60, 30.0),  4)
    g['Potassium_mg'][0] = round(max(cur['Potassium_mg'] * 0.50, 500.0), 4)
    g['Carb_g'][0]       = round(max(cur['Carb_g']       * 0.20, 30.0),  4)
    g['Fiber_g'][0]      = round(max(cur['Fiber_g']       * 0.30, 5.0),  4)

    # Safety check
    for f in features:
        if g[f][0] > g[f][1]:
            g[f] = [round(cur[f] * 0.85, 4), round(cur[f], 4)]

    return g

# %%
# ## 6. Generate Guardrail Ranges for All Groups × 3 Cases

all_guardrails = {}

# Categorical metadata (shared across all groups)
feature_metadata = {
    col: sorted(df_final[col].unique().tolist())
    for col in CAT_FEATURES if col in df_final.columns
}

for group_name, cfg in AGEGROUP_CONFIG.items():
    mdl = models.get(group_name)
    if mdl is None:
        print(f"\n[{group_name}] Model not available. Skipping.")
        continue

    indices = CASE_INDICES[group_name]

    for case_num, idx in enumerate(indices, start=1):
        case_key = f"{group_name}_case{case_num}"
        print(f"\n{'='*15} [{case_key}] Guardrail Generation {'='*15}")

        query = df_final.loc[[idx], X_FEATURES].copy()

        print(f"Patient profile — "
              f"BMI: {query.iloc[0]['BMI']:.2f}, "
              f"Energy: {query.iloc[0]['Energy_kcal']:.1f} kcal, "
              f"Sodium: {query.iloc[0]['Sodium_mg']:.1f} mg")

        # Step 1: LLM call (reasoning log only)
        print("Calling GPT-4o-mini guardrail agent...")
        agent_plan = get_clinical_guardrails(
            query, group_name, X_FEATURES, feature_metadata
        )
        if isinstance(agent_plan, list):
            agent_plan = agent_plan[0] if agent_plan else {}

        reasoning = str(agent_plan.get('reasoning', ''))
        print("Reasoning summary:", reasoning[:200], "...")

        # Step 2: Hard rules from patient baseline
        final_guardrails = apply_hard_rules(query, X_FEATURES)

        # Verify key ceiling constraints
        ceiling_vars = {'BMI', 'WaistCirc', 'Weight',
                        'Energy_kcal', 'Carb_g', 'Sugar_g', 'Sodium_mg'}
        print("\nKey guardrail ranges:")
        for k in ['BMI', 'Energy_kcal', 'Carb_g', 'Sodium_mg']:
            cur_val = float(query.iloc[0][k])
            lo, hi  = final_guardrails[k]
            flag    = "✓" if hi <= cur_val * 1.01 else "⚠"
            print(f"  {flag} {k:20s}: [{lo:.2f}, {hi:.2f}]  "
                  f"(current: {cur_val:.2f})")

        all_guardrails[case_key] = {
            'group':           group_name,
            'case':            case_num,
            'original_index':  idx,
            'patient_profile': query.to_dict(orient='records')[0],
            'llm_reasoning':   reasoning,
            'final_ranges':    final_guardrails,
        }

# %%
# ## 7. Save Guardrail Ranges

with open('../results/tables/guardrail_ranges.json', 'w', encoding='utf-8') as f:
    json.dump(all_guardrails, f, ensure_ascii=False, indent=4)

print(f"\n>>> Guardrail ranges saved: ../results/tables/guardrail_ranges.json")
print(f"    Total cases: {len(all_guardrails)}")
print(f"    Keys: {list(all_guardrails.keys())}")

Models loaded: ['MiddleAged_Male', 'MiddleAged_Female', 'Older_Male', 'Older_Female']
LLM model: gpt-4o-mini

=============== [MiddleAged_Male_case1] Guardrail Generation ===============
Patient profile — BMI: 27.59, Energy: 1934.6 kcal, Sodium: 2925.9 mg
Calling GPT-4o-mini guardrail agent...
Reasoning summary: {'CONFLICT_PREVENTION': 'If Sodium_mg is reduced below current value, then Carb_g must be capped at its current value (237.05 g) and Sugar_g at its current value (32.42 g). No change applied here.', ' ...

Key guardrail ranges:
  ✓ BMI                 : [23.45, 27.59]  (current: 27.59)
  ✓ Energy_kcal         : [1354.20, 1934.58]  (current: 1934.58)
  ✓ Carb_g              : [47.41, 237.05]  (current: 237.05)
  ✓ Sodium_mg           : [1755.56, 2633.34]  (current: 2925.93)

=============== [MiddleAged_Male_case2] Guardrail Generation ===============
Patient profile — BMI: 26.99, Energy: 1892.1 kcal, Sodium: 4818.4 mg
Calling GPT-4o-mini guardrail agent...
Reasoning summary: {'B

In [1]:
# =============================================================================
# [추가 셀] LLM-only Compliance Ablation
# 03_guardrail_agent.ipynb 최하단에 추가
# 목적: Hard Rule(Rules A-E) 적용 전 LLM 단독 출력의 G1-G4 준수율 측정
#       → Table 4 LLM-only 열 수치 산출
# =============================================================================

# %%
# ## [Ablation] 0. Libraries & Load Saved Guardrail Data

import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

# 기존 실행 결과 로드 (Section 7에서 저장한 파일)
with open('../results/tables/guardrail_ranges.json', 'r', encoding='utf-8') as f:
    all_guardrails = json.load(f)

agent_config = joblib.load('../results/tables/agent_config.pkl')
df_final     = joblib.load('../results/tables/df_final.pkl')

X_FEATURES = agent_config['X_features']

print(f"Loaded {len(all_guardrails)} cases: {list(all_guardrails.keys())}")

# %%
# ## [Ablation] 1. LLM Raw Ranges 재추출
# guardrail_ranges.json 에는 현재 llm_reasoning(텍스트)만 저장되어 있음
# → LLM을 재호출하여 guardrail_ranges dict를 확보

import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client    = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
LLM_MODEL = os.getenv('LLM_MODEL', 'gpt-4o-mini')

def get_llm_ranges_only(patient_data: pd.DataFrame,
                         group_name: str,
                         allowed_features: list,
                         metadata: dict) -> dict:
    """
    GPT-4o-mini 재호출: guardrail_ranges dict만 반환받음.
    temperature=0 으로 결정론적 출력 보장.
    """
    patient_info = patient_data.to_dict(orient='records')[0]

    prompt = f"""
You are a high-precision Clinical Data Strategist for a health insurance company.
You must respond ONLY in valid JSON using the exact variable names listed below.

[Patient Information ({group_name})]
{json.dumps(patient_info, ensure_ascii=False, indent=1)}

[Available Variable Names]
{", ".join(allowed_features)}

[Categorical Variable Valid Values]
{json.dumps(metadata, ensure_ascii=False, indent=1)}

[Guardrail Principles — apply ALL strictly]

1. CONFLICT PREVENTION:
   - Cap Carb_g at [0, current Carb_g] and Sugar_g at [0, current Sugar_g]
     whenever Sodium_mg is reduced below its current value.

2. PHYSICAL CONSISTENCY:
   - BMI, WaistCirc, Weight: all must move in the same direction.
   - Each bounded to [current * 0.85, current].

3. ENERGY FLOOR:
   - Energy_kcal: [max(current * 0.70, 500), current].

4. SAFETY: All lower bounds >= 0.

5. DIVERSITY HINT:
   - Allow Fat_g to vary in both directions for structural pathway diversity.

[Response Format — valid JSON only, no extra text]
{{
    "reasoning": "Detailed clinical rationale for each constraint",
    "guardrail_ranges": {{ "ExactVariableName": [min, max] }}
}}
"""

    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "You are an expert who responds strictly in valid JSON, "
                    "fully adhering to clinical guidelines and the provided "
                    "data schema."
                )
            },
            {"role": "user", "content": prompt},
        ],
        response_format={"type": "json_object"},
        temperature=0,
        top_p=1,
    )
    result = json.loads(response.choices[0].message.content)
    return result.get('guardrail_ranges', {})


# 재호출 실행
agent_config_full = joblib.load('../results/tables/agent_config.pkl')
CAT_FEATURES      = agent_config_full['cat_features']

feature_metadata = {
    col: sorted(df_final[col].unique().tolist())
    for col in CAT_FEATURES if col in df_final.columns
}

llm_raw_store = {}   # case_key → llm_only_ranges dict

for case_key, data in all_guardrails.items():
    group_name = data['group']
    idx        = data['original_index']
    query      = df_final.loc[[idx], X_FEATURES].copy()

    print(f"Calling LLM for {case_key} ...")
    raw_ranges = get_llm_ranges_only(
        query, group_name, X_FEATURES, feature_metadata
    )

    # float 정규화 (형식 불일치 방어)
    llm_only_ranges = {}
    for feat in X_FEATURES:
        rng = raw_ranges.get(feat)
        if isinstance(rng, (list, tuple)) and len(rng) == 2:
            try:
                llm_only_ranges[feat] = [float(rng[0]), float(rng[1])]
            except (TypeError, ValueError):
                llm_only_ranges[feat] = None
        else:
            llm_only_ranges[feat] = None

    llm_raw_store[case_key] = llm_only_ranges
    print(f"  → ranges extracted for "
          f"{sum(v is not None for v in llm_only_ranges.values())}"
          f"/{len(X_FEATURES)} features")

# %%
# ## [Ablation] 2. G1–G4 Compliance Check Functions (LLM-only)

def check_g1_llm(llm_ranges: dict, query: pd.DataFrame) -> bool:
    """
    G1 Physical consistency:
    BMI, WaistCirc, Weight 상한이 모두 현재값 이하 (감소 방향 일관성)
    """
    anthr = ['BMI', 'WaistCirc', 'Weight']
    dirs  = []
    for f in anthr:
        rng = llm_ranges.get(f)
        if rng is None:
            return False
        cur = float(query.iloc[0][f])
        dirs.append('down' if rng[1] < cur * 0.999 else 'same')
    return len(set(dirs)) == 1   # 모두 같은 방향이면 통과


def check_g2_llm(llm_ranges: dict, query: pd.DataFrame) -> bool:
    """
    G2 Energy safety floor:
    Energy 하한 >= max(0.70 * baseline, 500 kcal)
    """
    rng = llm_ranges.get('Energy_kcal')
    if rng is None:
        return False
    baseline       = float(query.iloc[0]['Energy_kcal'])
    required_floor = max(baseline * 0.70, 500.0)
    return rng[0] >= required_floor * 0.999   # 부동소수점 허용 오차


def check_g3_llm(llm_ranges: dict, query: pd.DataFrame) -> bool:
    """
    G3 Conflict prevention:
    Sodium 감소 시 Carb, Sugar 상한이 현재값 이하여야 함
    """
    sodium_rng = llm_ranges.get('Sodium_mg')
    if sodium_rng is None:
        return False
    cur_sodium     = float(query.iloc[0]['Sodium_mg'])
    sodium_reduced = sodium_rng[1] < cur_sodium * 0.999
    if not sodium_reduced:
        return True   # Sodium 감소 없으면 조건 해당 없음 → 통과
    for f in ['Carb_g', 'Sugar_g']:
        rng = llm_ranges.get(f)
        if rng is None:
            return False
        cur_val = float(query.iloc[0][f])
        if rng[1] > cur_val * 1.001:
            return False
    return True


def check_g4_llm(llm_ranges: dict, query: pd.DataFrame) -> bool:
    """
    G4 Macronutrient floors:
    Protein, Potassium, Carb, Fiber 하한 >= 가이드라인 기준
    """
    checks = {
        'Protein_g':    (0.60, 30.0),
        'Potassium_mg': (0.50, 500.0),
        'Carb_g':       (0.20, 30.0),
        'Fiber_g':      (0.30,  5.0),
    }
    for f, (ratio, abs_floor) in checks.items():
        rng = llm_ranges.get(f)
        if rng is None:
            return False
        cur_val  = float(query.iloc[0][f])
        required = max(cur_val * ratio, abs_floor)
        if rng[0] < required * 0.999:
            return False
    return True

# %%
# ## [Ablation] 3. Run Compliance Check for All 12 Cases

DIMS = ['G1', 'G2', 'G3', 'G4']
DIM_LABELS = {
    'G1': 'Physical consistency',
    'G2': 'Energy safety floor',
    'G3': 'Conflict prevention',
    'G4': 'Macronutrient floors',
}

results = []   # 행: case별 결과

for case_key, llm_only_ranges in llm_raw_store.items():
    data  = all_guardrails[case_key]
    idx   = data['original_index']
    query = df_final.loc[[idx], X_FEATURES].copy()

    compliance = {
        'G1': check_g1_llm(llm_only_ranges, query),
        'G2': check_g2_llm(llm_only_ranges, query),
        'G3': check_g3_llm(llm_only_ranges, query),
        'G4': check_g4_llm(llm_only_ranges, query),
    }

    row = {'case_key': case_key, 'group': data['group']}
    row.update({d: compliance[d] for d in DIMS})
    results.append(row)

    print(f"{case_key:30s}  "
          + "  ".join(f"{d}:{'✓' if compliance[d] else '✗'}"
                      for d in DIMS))

df_results = pd.DataFrame(results)

# %%
# ## [Ablation] 4. Aggregate: LLM-only Compliance Rate (Table 4 수치)

print("\n" + "="*55)
print("  LLM-only Compliance Rate (before Hard Rules A-E)")
print("  → Table 4 'LLM-only (%)' 열에 기입할 수치")
print("="*55)

total = len(df_results)
for d in DIMS:
    n_pass = df_results[d].sum()
    rate   = n_pass / total * 100
    print(f"  {d} {DIM_LABELS[d]:25s}: "
          f"{rate:5.1f}%  ({n_pass}/{total})")

print("="*55)
print(f"\n  (n = {total} cases, evaluated at case level)")

# %%
# ## [Ablation] 5. Save Results

out_path = '../results/tables/llm_only_compliance.json'
llm_compliance_summary = {
    'per_case': df_results.to_dict(orient='records'),
    'aggregate': {
        d: {
            'n_pass': int(df_results[d].sum()),
            'n_total': total,
            'rate_pct': round(df_results[d].sum() / total * 100, 1)
        }
        for d in DIMS
    }
}

with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(llm_compliance_summary, f, ensure_ascii=False, indent=4)

print(f"\n>>> Saved: {out_path}")
print("    Use aggregate rates to fill Table 4 LLM-only column.")

Loaded 12 cases: ['MiddleAged_Male_case1', 'MiddleAged_Male_case2', 'MiddleAged_Male_case3', 'MiddleAged_Female_case1', 'MiddleAged_Female_case2', 'MiddleAged_Female_case3', 'Older_Male_case1', 'Older_Male_case2', 'Older_Male_case3', 'Older_Female_case1', 'Older_Female_case2', 'Older_Female_case3']
Calling LLM for MiddleAged_Male_case1 ...
  → ranges extracted for 31/31 features
Calling LLM for MiddleAged_Male_case2 ...
  → ranges extracted for 31/31 features
Calling LLM for MiddleAged_Male_case3 ...
  → ranges extracted for 31/31 features
Calling LLM for MiddleAged_Female_case1 ...
  → ranges extracted for 31/31 features
Calling LLM for MiddleAged_Female_case2 ...
  → ranges extracted for 31/31 features
Calling LLM for MiddleAged_Female_case3 ...
  → ranges extracted for 7/31 features
Calling LLM for Older_Male_case1 ...
  → ranges extracted for 31/31 features
Calling LLM for Older_Male_case2 ...
  → ranges extracted for 31/31 features
Calling LLM for Older_Male_case3 ...
  → ranges e

In [2]:
# 노트북에서 확인
print(llm_raw_store['MiddleAged_Female_case3'])

{'BMI': [22.667885, 26.66751407237417], 'WaistCirc': [72.08, 84.8], 'Weight': [52.685, 62.1], 'Energy_kcal': [1038.3894176991898, 1484.8420281416998], 'Carb_g': [0.0, 214.6201403440997], 'Sugar_g': [0.0, 41.61512985678249], 'Sodium_mg': None, 'Fat_g': [0.0, 49.265708330974455], 'SaturatedFat_g': None, 'Fiber_g': None, 'Potassium_mg': None, 'Protein_g': None, 'ObesityStatus': None, 'WeightChangeStatus': None, 'WeightLossAmount': None, 'WeightGainAmount': None, 'DrinkingFrequency': None, 'DrinkingAmount': None, 'SmokingStatus': None, 'VigorousActivity_Work': None, 'VigorousActivity_Leisure': None, 'ModerateActivity_Work': None, 'WalkingActivity': None, 'AerobicActivityRate': None, 'BreakfastFrequency': None, 'StressLevel': None, 'StressAwarenessRate': None, 'PersonalIncomeQuartile': None, 'HouseholdIncomeQuartile': None, 'EducationLevel': None, 'HealthScreeningStatus': None}
